# New Cluster Analysis
Reads final hotel clusters from `s3a://delta-lake-mumbai/bronze/final_clusters/` and explores their structure and statistics.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, round as spark_round, min as spark_min, max as spark_max
import os

# Override any problematic environment variables
os.environ.pop('HADOOP_CONF_DIR', None)
os.environ.pop('YARN_CONF_DIR', None)

# Create Spark session with MinIO/S3A configuration
# Simplified without Delta Lake dependencies - read as parquet directly
spark = SparkSession.builder \
    .appName("Analyze Hotel Pairs") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.socket.send.buffer", "8192") \
    .config("spark.hadoop.fs.s3a.socket.recv.buffer", "8192") \
    .config("spark.hadoop.fs.s3a.attempts.maximum", "3") \
    .config("spark.hadoop.fs.s3a.retry.limit", "3") \
    .config("spark.hadoop.fs.s3a.retry.interval", "500") \
    .config("spark.hadoop.fs.s3a.retry.throttle.limit", "3") \
    .config("spark.hadoop.fs.s3a.retry.throttle.interval", "1000") \
    .config("spark.hadoop.fs.s3a.connection.maximum", "50") \
    .config("spark.hadoop.fs.s3a.threads.max", "10") \
    .config("spark.hadoop.fs.s3a.threads.core", "5") \
    .config("spark.hadoop.fs.s3a.threads.keepalivetime", "60") \
    .config("spark.hadoop.fs.s3a.max.total.tasks", "5") \
    .config("spark.hadoop.fs.s3a.readahead.range", "65536") \
    .config("spark.hadoop.fs.s3a.paging.maximum", "5") \
    .config("spark.hadoop.fs.s3a.list.version", "2") \
    .config("spark.hadoop.fs.s3a.committer.threads", "4") \
    .config("spark.hadoop.fs.s3a.committer.staging.tmp.path", "/tmp/staging") \
    .config("spark.hadoop.fs.s3a.buffer.dir", "/tmp") \
    .config("spark.hadoop.fs.s3a.multipart.size", "104857600") \
    .config("spark.hadoop.fs.s3a.multipart.threshold", "2147483647") \
    .config("spark.hadoop.fs.s3a.multipart.purge.age", "86400") \
    .config("spark.hadoop.fs.s3a.fast.upload", "true") \
    .config("spark.hadoop.fs.s3a.fast.upload.buffer", "disk") \
    .config("spark.hadoop.fs.s3a.fast.upload.active.blocks", "4") \
    .config("spark.hadoop.fs.s3a.block.size", "33554432") \
    .config("spark.hadoop.fs.s3a.metadatastore.authoritative", "false") \
    .config("spark.sql.files.maxPartitionBytes", "134217728") \
    .config("spark.driver.memory", "2g") \
    .config("spark.ui.port", "4050") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("✓ Spark session created successfully")
print("✓ Spark UI available at: http://localhost:4050")

:: loading settings :: url = jar:file:/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/nakul.patil/.ivy2.5.2/cache
The jars for the packages stored in: /Users/nakul.patil/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-29224178-e7a2-4dcb-96c8-b49fb3ee95d9;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in local-m2-cache
:: resolution report :: resolve 105ms :: artifacts dl 3ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final

✓ Spark session created successfully
✓ Spark UI available at: http://localhost:4050


## 2. Load Final Clusters

In [7]:
CLUSTERS_PATH = "s3a://delta-lake-mumbai/bronze/final_clusters/"

df_clusters = spark.read.parquet(CLUSTERS_PATH)

print(f"✓ Loaded clusters from: {CLUSTERS_PATH}")
print(f"  Total rows : {df_clusters.count():,}")
print(f"  Columns    : {len(df_clusters.columns)}")
print()
df_clusters.printSchema()

✓ Loaded clusters from: s3a://delta-lake-mumbai/bronze/final_clusters/
  Total rows : 2,583
  Columns    : 4

root
 |-- cluster_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- uid: string (nullable = true)
 |-- providerName: string (nullable = true)



## 3. Preview Data

In [8]:
df_clusters.show(20, truncate=False)

+----------+--------------------------------------------------------------------+----------------------------------------------------------------+------------+
|cluster_id|name                                                                |uid                                                             |providerName|
+----------+--------------------------------------------------------------------+----------------------------------------------------------------+------------+
|1633      |zenith homes - powai suites                                         |035b8c59ae5fe2f9c168f88750f579347b8e21e2054f44d6f144e8fbb53ce671|bookingcom  |
|1634      |hotel boston                                                        |16474e7c4c1269d144e0023542513e9a263faa9751467dbbb0bcbf49b08c5e57|bookingcom  |
|1635      |onn shelter inn                                                     |33d728db66bd289a7d9d45ea7c87eb047385417e208500744c5af80b4893d33b|bookingcom  |
|1636      |super townhouse oak vashi   

## 4. Cluster Statistics

In [9]:
from pyspark.sql.functions import countDistinct, size

# Detect the cluster ID column (common names: cluster_id, cluster, group_id)
cluster_col_candidates = [c for c in df_clusters.columns if 'cluster' in c.lower() or c.lower() in ('group_id', 'entity_id')]
cluster_col = cluster_col_candidates[0] if cluster_col_candidates else df_clusters.columns[0]
print(f"Using '{cluster_col}' as cluster ID column\n")

# Number of distinct clusters
n_clusters = df_clusters.select(countDistinct(col(cluster_col))).collect()[0][0]
total_rows  = df_clusters.count()

print(f"Total clusters : {n_clusters:,}")
print(f"Total records  : {total_rows:,}")
print(f"Avg cluster size: {total_rows / n_clusters:.2f}" if n_clusters else "")

# Cluster size distribution
cluster_sizes = (
    df_clusters
    .groupBy(cluster_col)
    .agg(count("*").alias("cluster_size"))
)

cluster_sizes.select(
    spark_min("cluster_size").alias("min_size"),
    spark_max("cluster_size").alias("max_size"),
    avg("cluster_size").alias("avg_size"),
).show()

print("Cluster size distribution:")
cluster_sizes.groupBy("cluster_size").agg(count("*").alias("num_clusters")) \
    .orderBy("cluster_size").show(30)

Using 'cluster_id' as cluster ID column



Total clusters : 2,583
Total records  : 2,583
Avg cluster size: 1.00


+--------+--------+--------+
|min_size|max_size|avg_size|
+--------+--------+--------+
|       1|       1|     1.0|
+--------+--------+--------+

Cluster size distribution:


+------------+------------+
|cluster_size|num_clusters|
+------------+------------+
|           1|        2583|
+------------+------------+



## 5. Inspect a Sample Cluster

In [10]:
# Pick the largest cluster and show all its members
largest_cluster_id = (
    cluster_sizes.orderBy(col("cluster_size").desc()).first()[cluster_col]
)
print(f"Largest cluster id: {largest_cluster_id}")
df_clusters.filter(col(cluster_col) == largest_cluster_id).show(truncate=False)

Largest cluster id: 2294
+----------+----------+----------------------------------------------------------------+------------+
|cluster_id|name      |uid                                                             |providerName|
+----------+----------+----------------------------------------------------------------+------------+
|2294      |ruby hotel|b43fc1d934a6b13608648d7d34f4760eef83c805deed629a25e70673068d2602|bookingcom  |
+----------+----------+----------------------------------------------------------------+------------+



In [ ]:
mapped_path = f"s3a://delta-lake-mumbai/bronze/hotels"
print(f"Reading from: {mapped_path}")

# Read Parquet data
df = spark.read.parquet(mapped_path)

# Display basic info
print(f"\n✓ Data loaded successfully")
print(f"Total records: {df.count():,}")
print(f"Total columns: {len(df.columns)}")

df.printSchema()
df.show(20, truncate=False)


Reading from: s3a://delta-lake-mumbai/bronze/hotels

✓ Data loaded successfully
Total records: 2,584
Total columns: 35
root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- relevanceScore: string (nullable = true)
 |-- providerId: string (nullable = true)
 |-- providerHotelId: string (nullable = true)
 |-- providerName: string (nullable = true)
 |-- language: string (nullable = true)
 |-- geoCode: struct (nullable = true)
 |    |-- lat: string (nullable = true)
 |    |-- long: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- address: struct (nullable = true)
 |    |    |-- line1: string (nullable = true)
 |    |    |-- city: struct (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |-- state: struct (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |-- country: struct (nullable = true)
 |    |    |    |-- code: string (nullable = true)
 |    |    |    |-- name: string (nullable

26/03/07 14:59:50 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------+-----------------------------------------------------------------+--------------+----------+---------------+------------+--------+----------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+---------+----------+--------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+------------------+-----------+------------+---------------------------------------------------------------------------------

26/03/07 21:28:40 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 905109 ms exceeds timeout 120000 ms
26/03/07 21:28:40 WARN SparkContext: Killing executors is not supported by current scheduler.
26/03/07 21:28:45 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

In [12]:

mapped_path = f"s3a://delta-lake-mumbai/bronze/comparison_audit_log"
print(f"Reading from: {mapped_path}")

# Read Parquet data
df = spark.read.parquet(mapped_path)

# Display basic info
print(f"\n✓ Data loaded successfully")
print(f"Total records: {df.count():,}")
print(f"Total columns: {len(df.columns)}")

df.printSchema()
df.show(20, truncate=False)

Reading from: s3a://delta-lake-mumbai/bronze/comparison_audit_log

✓ Data loaded successfully
Total records: 27,266
Total columns: 25
root
 |-- hotel_id: string (nullable = true)
 |-- hotel_name: string (nullable = true)
 |-- compared_with_id: string (nullable = true)
 |-- compared_with_name: string (nullable = true)
 |-- score: double (nullable = true)
 |-- status: string (nullable = true)
 |-- is_matched: boolean (nullable = true)
 |-- failed_conditions: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- comparison_at: timestamp (nullable = true)
 |-- geo_distance_km: double (nullable = true)
 |-- name_score_jaccard: float (nullable = true)
 |-- normalized_name_score_jaccard: float (nullable = true)
 |-- name_score_lcs: float (nullable = true)
 |-- normalized_name_score_lcs: float (nullable = true)
 |-- name_score_levenshtein: float (nullable = true)
 |-- normalized_name_score_levenshtein: float (nullable = true)
 |-- name_score_sbert: float (nullable = true